# Chapter 3: Claim Development Data

Code of Chapter 3 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## 3.1.2 Accident year and development age

In [ ]:
from gitriangle import development_age
development_age(2023, "2025-12-31")        # 36
development_age(2025, "2025-06-30")        # 6

## 3.2.3 The example file

In [ ]:
import pandas as pd
from gitriangle import triangle, to_incremental, to_cumulative, calendar_diagonal

claims = pd.DataFrame({
    "claim_id": ["K1", "K2", "K3", "K4", "K5", "K6"],
    "accident_date": ["2023-03-10", "2023-11-20", "2023-07-01", "2024-02-14", "2024-12-20", "2025-05-05"],
    "report_date":   ["2023-03-15", "2024-01-10", "2023-07-05", "2024-02-20", "2025-01-08", "2025-05-06"],
})
rows = [("K1", "2023-03-15", "reserve", 5000), ("K1", "2023-09-01", "payment", 2000),
        ("K1", "2023-09-01", "reserve", 3500), ("K1", "2024-05-01", "payment", 3800),
        ("K1", "2024-05-01", "reserve", 0),
        ("K2", "2024-01-10", "reserve", 4000), ("K2", "2024-06-30", "payment", 1500),
        ("K2", "2024-06-30", "reserve", 3000), ("K2", "2025-04-01", "payment", 3000),
        ("K2", "2025-04-01", "reserve", 0),
        ("K3", "2023-07-05", "reserve", 1000), ("K3", "2023-08-01", "payment", 900),
        ("K3", "2023-08-01", "reserve", 0),
        ("K4", "2024-02-20", "reserve", 6000), ("K4", "2024-10-01", "payment", 2500),
        ("K4", "2024-10-01", "reserve", 4000), ("K4", "2025-06-01", "reserve", 4500),
        ("K5", "2025-01-08", "reserve", 2000), ("K5", "2025-03-01", "payment", 2200),
        ("K5", "2025-03-01", "reserve", 0),
        ("K6", "2025-05-06", "reserve", 3000), ("K6", "2025-11-15", "payment", 1000),
        ("K6", "2025-11-15", "reserve", 2500)]
tx = pd.DataFrame(rows, columns=["claim_id", "date", "type", "amount"])

## 3.2.4 Worked example: the cumulative paid triangle

In [ ]:
P = triangle(claims, tx, "paid", first_ay=2023, last_year=2025)
X = to_incremental(P)
calendar_diagonal(X, 2025).sum()           # 6200.0
to_cumulative(X).equals(P)                 # True

## 3.4.1 Five underwriting years

In [ ]:
from gitriangle import synthetic_history, exposure_by_accident_year
pol, cl, txs = synthetic_history(first_year=2020, last_year=2025, seed=100)
len(pol), len(cl), len(txs)                # 34009 policies, 3506 claims, 21298 transactions

## 3.4.2 The paid and incurred triangles

In [ ]:
P = triangle(cl, txs, "paid", 2021, 2025)
I = triangle(cl, txs, "incurred", 2021, 2025)

## 3.4.3 Diagnostic ratios

In [ ]:
R = triangle(cl, txs, "reported_count", 2021, 2025)
C = triangle(cl, txs, "closed_count", 2021, 2025)
O = triangle(cl, txs, "open_count", 2021, 2025)
CR = triangle(cl, txs, "case_reserve", 2021, 2025)
(P / I).round(3)                           # paid-to-incurred
(C / R).round(3)                           # closure rate
(CR / O).round(0)                          # average case reserve

from gidevelop import age_to_age           # age-to-age factors (Chapter 4)
X = to_incremental(P)
for ay in X.index:                         # a 10% backlog cleared in calendar year 2025
    X.loc[ay, 12 * (2025 - ay + 1)] *= 1.10
age_to_age(to_cumulative(X)).round(4)      # latest factors 1.0077, 1.0261, 1.1560, 1.8888
calendar_diagonal(to_incremental(P), 2025).sum()   # 2883450.0 paid in 2025

## 3.4.4 Exposure and frequency

In [ ]:
E = exposure_by_accident_year(pol, range(2021, 2026))
(R[12] / E["earned_exposure"]).round(4)    # 0.0947, 0.0962, 0.0928, 0.0904, 0.0955

## 3.5 Triangles in Other Software

In [ ]:
from gitriangle import to_long
long_paid = to_long(P, "paid")
long_paid.head(3)
#   accident_year  valuation    paid
# 0    2021-01-01 2021-12-31  1032119.41
# 1    2021-01-01 2022-12-31  1932400.71
# 2    2021-01-01 2023-12-31  2254031.80

In [ ]:
try:
    import chainladder  # optional: not available in the browser version
except ImportError:
    chainladder = None
    print("chainladder is not installed: this cross-check is skipped.")
if chainladder is not None:
    import chainladder as chl
    tri = chl.Triangle(long_paid, origin="accident_year", development="valuation",
                      columns=["paid"], cumulative=True)
    tri.to_frame().columns.tolist()            # [12, 24, 36, 48, 60]

In [ ]:
try:
    import chainladder  # optional: not available in the browser version
except ImportError:
    chainladder = None
    print("chainladder is not installed: this cross-check is skipped.")
if chainladder is not None:
    import chainladder as chl
    clrd = chl.load_sample("clrd")              # 775 company-and-line triangles
    pp = clrd.groupby("LOB").sum().loc["ppauto"]["CumPaidLoss"].to_frame()
    pp.index = pp.index.year                    # accident years 1988-1997, ages 12-120
    pp.iloc[:3, :3]
    #            12         24         36
    # 1988  3092818.0  5942711.0  7239089.0
    # 1989  3556683.0  6753435.0  8219551.0
    # 1990  4015052.0  7478257.0  9094949.0

## Exercise 1: Count triangles by hand

In [ ]:
triangle(claims, tx, "open_count", 2023, 2025)

## Exercise 2: Payments of calendar year 2024

In [ ]:
P6 = triangle(claims, tx, "paid", 2023, 2025)     # the six-claim example
calendar_diagonal(to_incremental(P6), 2024).sum()  # 7800.0

## 3.8 Python Walkthrough

In [ ]:
from gitriangle import (development_age, triangle, to_incremental, to_cumulative,
                        calendar_diagonal, synthetic_history, exposure_by_accident_year, to_long)

# Section 3.2: the six-claim example
P6 = triangle(claims, tx, "paid", 2023, 2025)              # 2900 / 8200 / 11200 ...
I6 = triangle(claims, tx, "incurred", 2023, 2025)          # 6400 / 11200 / 11200 ...
calendar_diagonal(to_incremental(P6), 2025).sum()           # 6200.0

# Section 3.4: the synthetic portfolio, accident years 2021-2025
pol, cl, txs = synthetic_history(2020, 2025, seed=100)
P = triangle(cl, txs, "paid", 2021, 2025)
I = triangle(cl, txs, "incurred", 2021, 2025)
R = triangle(cl, txs, "reported_count", 2021, 2025)
(P / I).loc[2023, 12].round(3)                              # 0.582
E = exposure_by_accident_year(pol, range(2021, 2026))
E["earned_exposure"].round(2).tolist()                      # 5120.07 ... 6212.18